In [0]:
# MVP Engenharia de Dados
# ETAPA 01 - Ingestão da camada Bronze

from pyspark.sql import functions as F

SOURCE_PATH = (
    "/Volumes/workspace/mvp_raw/input_files/"
    "student_productivity_distraction_dataset_20000.csv"
)

BRONZE_TABLE = "workspace.mvp_bronze.student_productivity_raw"

print("Arquivo de origem:")
print(SOURCE_PATH)

print("\nTabela de destino:")
print(BRONZE_TABLE)

Arquivo de origem:
/Volumes/workspace/mvp_raw/input_files/student_productivity_distraction_dataset_20000.csv

Tabela de destino:
workspace.mvp_bronze.student_productivity_raw


In [0]:
# Leitura inicial do arquivo CSV
# Nesta etapa ainda não alteramos nem persistimos os dados.

df_raw = (
    spark.read
    .option("header", True)
    .option("inferSchema", False)
    .option("mode", "PERMISSIVE")
    .csv(SOURCE_PATH)
)

# Quantidade de registros e colunas
total_linhas = df_raw.count()
total_colunas = len(df_raw.columns)

print(f"Quantidade de registros: {total_linhas}")
print(f"Quantidade de colunas: {total_colunas}")

print("\nColunas encontradas:")
for coluna in df_raw.columns:
    print("-", coluna)

Quantidade de registros: 5999
Quantidade de colunas: 18

Colunas encontradas:
- student_id
- age
- gender
- study_hours_per_day
- sleep_hours
- phone_usage_hours
- social_media_hours
- youtube_hours
- gaming_hours
- breaks_per_day
- coffee_intake_mg
- exercise_minutes
- assignments_completed
- attendance_percentage
- stress_level
- focus_score
- final_grade
- productivity_score


In [0]:
# Validação do dataset antes da persistência na Bronze

print("=== RESUMO DO DATASET BRUTO ===")
print(f"Quantidade de registros: {df_raw.count()}")
print(f"Quantidade de colunas: {len(df_raw.columns)}")

print("\n=== SCHEMA ORIGINAL ===")
df_raw.printSchema()

print("\n=== PRIMEIROS REGISTROS ===")
display(df_raw.limit(10))

=== RESUMO DO DATASET BRUTO ===
Quantidade de registros: 5999
Quantidade de colunas: 18

=== SCHEMA ORIGINAL ===
root
 |-- student_id: string (nullable = true)
 |-- age: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- study_hours_per_day: string (nullable = true)
 |-- sleep_hours: string (nullable = true)
 |-- phone_usage_hours: string (nullable = true)
 |-- social_media_hours: string (nullable = true)
 |-- youtube_hours: string (nullable = true)
 |-- gaming_hours: string (nullable = true)
 |-- breaks_per_day: string (nullable = true)
 |-- coffee_intake_mg: string (nullable = true)
 |-- exercise_minutes: string (nullable = true)
 |-- assignments_completed: string (nullable = true)
 |-- attendance_percentage: string (nullable = true)
 |-- stress_level: string (nullable = true)
 |-- focus_score: string (nullable = true)
 |-- final_grade: string (nullable = true)
 |-- productivity_score: string (nullable = true)


=== PRIMEIROS REGISTROS ===


student_id,age,gender,study_hours_per_day,sleep_hours,phone_usage_hours,social_media_hours,youtube_hours,gaming_hours,breaks_per_day,coffee_intake_mg,exercise_minutes,assignments_completed,attendance_percentage,stress_level,focus_score,final_grade,productivity_score
1,23,Female,4.35,3.63,3.38,2.73,1.83,5.26,6,347,111,2,57.21,10,57,81.87,33.78
2,20,Male,6.14,6.58,5.48,1.51,3.13,1.73,13,403,28,10,91.27,10,49,60.9,48.99
3,29,Female,4.98,3.26,4.83,3.63,0.18,4.71,1,419,102,8,63.14,2,38,86.22,36.6
4,27,Female,3.19,4.58,10.06,3.95,5.75,2.52,9,178,28,18,40.51,6,50,71.77,19.87
5,24,Male,7.67,6.21,3.02,1.59,5.46,5.65,8,436,105,7,45.53,6,41,90.13,52.9
6,29,Other,7.18,3.52,4.02,3.74,1.42,0.16,10,392,12,3,47.58,10,70,59.48,47.31
7,21,Female,9.06,6.36,11.45,5.99,2.2,4.44,14,87,28,15,43.5,8,35,62.71,41.23
8,23,Female,6.37,4.86,3.31,1.37,4.36,5.13,2,152,103,17,75.22,6,59,52.22,53.81
9,26,Male,4.19,4.87,9.66,2.87,0.1,3.38,13,460,42,11,44.79,3,39,76.15,25.99
10,19,Female,7.28,9.56,2.13,0.81,1.35,2.55,7,416,107,6,79.15,10,73,88.53,73.18


In [0]:
# Persistência da camada Bronze
# Mantemos os dados como recebidos e adicionamos apenas metadados de rastreabilidade.

from pyspark.sql import functions as F

df_bronze = (
    df_raw
    .withColumn("_ingestion_ts", F.current_timestamp())
    .withColumn("_source_file", F.col("_metadata.file_path"))
)

print("Registros antes da gravação:", df_bronze.count())
print("Colunas após inclusão dos metadados:", len(df_bronze.columns))

display(df_bronze.limit(10))

Registros antes da gravação: 5999
Colunas após inclusão dos metadados: 20


student_id,age,gender,study_hours_per_day,sleep_hours,phone_usage_hours,social_media_hours,youtube_hours,gaming_hours,breaks_per_day,coffee_intake_mg,exercise_minutes,assignments_completed,attendance_percentage,stress_level,focus_score,final_grade,productivity_score,_ingestion_ts,_source_file
1,23,Female,4.35,3.63,3.38,2.73,1.83,5.26,6,347,111,2,57.21,10,57,81.87,33.78,2026-08-28T18:37:08.653Z,dbfs:/Volumes/workspace/mvp_raw/input_files/student_productivity_distraction_dataset_20000.csv
2,20,Male,6.14,6.58,5.48,1.51,3.13,1.73,13,403,28,10,91.27,10,49,60.9,48.99,2026-08-28T18:37:08.653Z,dbfs:/Volumes/workspace/mvp_raw/input_files/student_productivity_distraction_dataset_20000.csv
3,29,Female,4.98,3.26,4.83,3.63,0.18,4.71,1,419,102,8,63.14,2,38,86.22,36.6,2026-08-28T18:37:08.653Z,dbfs:/Volumes/workspace/mvp_raw/input_files/student_productivity_distraction_dataset_20000.csv
4,27,Female,3.19,4.58,10.06,3.95,5.75,2.52,9,178,28,18,40.51,6,50,71.77,19.87,2026-08-28T18:37:08.653Z,dbfs:/Volumes/workspace/mvp_raw/input_files/student_productivity_distraction_dataset_20000.csv
5,24,Male,7.67,6.21,3.02,1.59,5.46,5.65,8,436,105,7,45.53,6,41,90.13,52.9,2026-08-28T18:37:08.653Z,dbfs:/Volumes/workspace/mvp_raw/input_files/student_productivity_distraction_dataset_20000.csv
6,29,Other,7.18,3.52,4.02,3.74,1.42,0.16,10,392,12,3,47.58,10,70,59.48,47.31,2026-08-28T18:37:08.653Z,dbfs:/Volumes/workspace/mvp_raw/input_files/student_productivity_distraction_dataset_20000.csv
7,21,Female,9.06,6.36,11.45,5.99,2.2,4.44,14,87,28,15,43.5,8,35,62.71,41.23,2026-08-28T18:37:08.653Z,dbfs:/Volumes/workspace/mvp_raw/input_files/student_productivity_distraction_dataset_20000.csv
8,23,Female,6.37,4.86,3.31,1.37,4.36,5.13,2,152,103,17,75.22,6,59,52.22,53.81,2026-08-28T18:37:08.653Z,dbfs:/Volumes/workspace/mvp_raw/input_files/student_productivity_distraction_dataset_20000.csv
9,26,Male,4.19,4.87,9.66,2.87,0.1,3.38,13,460,42,11,44.79,3,39,76.15,25.99,2026-08-28T18:37:08.653Z,dbfs:/Volumes/workspace/mvp_raw/input_files/student_productivity_distraction_dataset_20000.csv
10,19,Female,7.28,9.56,2.13,0.81,1.35,2.55,7,416,107,6,79.15,10,73,88.53,73.18,2026-08-28T18:37:08.653Z,dbfs:/Volumes/workspace/mvp_raw/input_files/student_productivity_distraction_dataset_20000.csv


In [0]:
BRONZE_TABLE = "workspace.mvp_bronze.student_productivity_raw"

(
    df_bronze.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(BRONZE_TABLE)
)

print("Tabela Bronze persistida com sucesso:")
print(BRONZE_TABLE)

Tabela Bronze persistida com sucesso:
workspace.mvp_bronze.student_productivity_raw


In [0]:
df_bronze_check = spark.table(BRONZE_TABLE)

print("=== VALIDAÇÃO DA BRONZE ===")
print("Tabela:", BRONZE_TABLE)
print("Quantidade de registros:", df_bronze_check.count())
print("Quantidade de colunas:", len(df_bronze_check.columns))

display(df_bronze_check.limit(10))

=== VALIDAÇÃO DA BRONZE ===
Tabela: workspace.mvp_bronze.student_productivity_raw
Quantidade de registros: 5999
Quantidade de colunas: 20


student_id,age,gender,study_hours_per_day,sleep_hours,phone_usage_hours,social_media_hours,youtube_hours,gaming_hours,breaks_per_day,coffee_intake_mg,exercise_minutes,assignments_completed,attendance_percentage,stress_level,focus_score,final_grade,productivity_score,_ingestion_ts,_source_file
1,23,Female,4.35,3.63,3.38,2.73,1.83,5.26,6,347,111,2,57.21,10,57,81.87,33.78,2026-08-28T18:37:39.253Z,dbfs:/Volumes/workspace/mvp_raw/input_files/student_productivity_distraction_dataset_20000.csv
2,20,Male,6.14,6.58,5.48,1.51,3.13,1.73,13,403,28,10,91.27,10,49,60.9,48.99,2026-08-28T18:37:39.253Z,dbfs:/Volumes/workspace/mvp_raw/input_files/student_productivity_distraction_dataset_20000.csv
3,29,Female,4.98,3.26,4.83,3.63,0.18,4.71,1,419,102,8,63.14,2,38,86.22,36.6,2026-08-28T18:37:39.253Z,dbfs:/Volumes/workspace/mvp_raw/input_files/student_productivity_distraction_dataset_20000.csv
4,27,Female,3.19,4.58,10.06,3.95,5.75,2.52,9,178,28,18,40.51,6,50,71.77,19.87,2026-08-28T18:37:39.253Z,dbfs:/Volumes/workspace/mvp_raw/input_files/student_productivity_distraction_dataset_20000.csv
5,24,Male,7.67,6.21,3.02,1.59,5.46,5.65,8,436,105,7,45.53,6,41,90.13,52.9,2026-08-28T18:37:39.253Z,dbfs:/Volumes/workspace/mvp_raw/input_files/student_productivity_distraction_dataset_20000.csv
6,29,Other,7.18,3.52,4.02,3.74,1.42,0.16,10,392,12,3,47.58,10,70,59.48,47.31,2026-08-28T18:37:39.253Z,dbfs:/Volumes/workspace/mvp_raw/input_files/student_productivity_distraction_dataset_20000.csv
7,21,Female,9.06,6.36,11.45,5.99,2.2,4.44,14,87,28,15,43.5,8,35,62.71,41.23,2026-08-28T18:37:39.253Z,dbfs:/Volumes/workspace/mvp_raw/input_files/student_productivity_distraction_dataset_20000.csv
8,23,Female,6.37,4.86,3.31,1.37,4.36,5.13,2,152,103,17,75.22,6,59,52.22,53.81,2026-08-28T18:37:39.253Z,dbfs:/Volumes/workspace/mvp_raw/input_files/student_productivity_distraction_dataset_20000.csv
9,26,Male,4.19,4.87,9.66,2.87,0.1,3.38,13,460,42,11,44.79,3,39,76.15,25.99,2026-08-28T18:37:39.253Z,dbfs:/Volumes/workspace/mvp_raw/input_files/student_productivity_distraction_dataset_20000.csv
10,19,Female,7.28,9.56,2.13,0.81,1.35,2.55,7,416,107,6,79.15,10,73,88.53,73.18,2026-08-28T18:37:39.253Z,dbfs:/Volumes/workspace/mvp_raw/input_files/student_productivity_distraction_dataset_20000.csv
